# 2일차 4교시 · 교차 검증 (코드 없이)

각 칸의 ▶ 버튼을 누르고, 오른쪽 슬라이더와 목록만 바꾸면 됩니다. 코드는 숨겨 두었습니다.

1. 한 번의 분할 결과가 얼마나 **흔들리는지** 확인합니다.
2. KFold와 StratifiedKFold의 차이를 봅니다.
3. 교차 검증 결과를 **평균 ± 표준편차**로 읽습니다.
4. [선택] 교차 검증에서도 **데이터 누수**가 생기는 것을 봅니다.

그래프 글자는 Colab에서 한글이 깨지지 않도록 영어로 적었습니다.

In [ ]:
#@title 0. 준비 (▶)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_validate, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

X, y = load_breast_cancer(return_X_y=True)
print("준비 완료")

## 1. 한 번의 분할은 운이다

데이터를 학습용과 테스트용으로 나누는 방법만 바꿔 가며 **같은 모델**의 정확도를 여러 번 측정합니다.

In [ ]:
#@title 1. 분할을 바꿔 가며 정확도 측정
측정_횟수 = 20 #@param {type:"slider", min:5, max:50, step:5}
트리_깊이 = 3 #@param {type:"slider", min:1, max:10, step:1}
scores = []
for seed in range(측정_횟수):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=seed, stratify=y)
    scores.append(DecisionTreeClassifier(max_depth=트리_깊이, random_state=0).fit(X_tr, y_tr).score(X_te, y_te))
print(f"최저 {min(scores):.3f}, 최고 {max(scores):.3f}, 평균 {np.mean(scores):.3f}")

plt.figure(figsize=(6, 3))
plt.plot(range(측정_횟수), scores, marker="o")
plt.xlabel("split random_state")
plt.ylabel("test accuracy")
plt.title("Same model, different splits")
plt.show()

같은 모델인데 분할에 따라 정확도가 달라집니다. 숫자 하나로 모델을 판단하면 위험한 이유입니다.

PM 질문: 개발팀이 "정확도 0.97 나왔습니다"라고 보고하면, **어떤 질문**을 해야 할까요?

## 2. KFold vs StratifiedKFold

데이터를 **정렬된 상태**로 두고 나누면 차이가 뚜렷합니다. 각 폴드의 검증 세트에 악성(0)이 몇 %인지 봅니다.

In [ ]:
#@title 2. 폴드별 악성 비율 비교
order = np.argsort(y)                      # 라벨 순으로 정렬 (악성이 앞쪽)
X_sorted, y_sorted = X[order], y[order]
rows = []
for name, cv in [("KFold (섞지 않음)", KFold(5)), ("StratifiedKFold", StratifiedKFold(5))]:
    for i, (_, val_idx) in enumerate(cv.split(X_sorted, y_sorted)):
        rows.append({"방법": name, "폴드": i + 1,
                     "검증 세트 악성 비율": round((y_sorted[val_idx] == 0).mean(), 2)})
pd.DataFrame(rows).pivot(index="폴드", columns="방법", values="검증 세트 악성 비율")

원본 악성 비율은 약 0.37입니다. 섞지 않은 KFold는 어떤 폴드에는 악성만, 어떤 폴드에는 양성 종양만 들어갑니다. **분류 문제에는 StratifiedKFold**를 씁니다.

## 3. 교차 검증 결과 읽기

데이터를 K개로 나눠 K번 평가하고 **평균 ± 표준편차**로 보고합니다. K와 모델을 바꿔 보세요.

In [ ]:
#@title 3. 교차 검증 실행
K = "5" #@param ["3", "5", "10"]
모델 = "로지스틱 회귀" #@param ["로지스틱 회귀", "결정 트리(깊이 3)", "결정 트리(제한 없음)"]
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
models = {
    "로지스틱 회귀": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "결정 트리(깊이 3)": DecisionTreeClassifier(max_depth=3, random_state=0),
    "결정 트리(제한 없음)": DecisionTreeClassifier(random_state=0),
}
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, pos_label=0),   # 악성 = 양성 클래스
    "recall": make_scorer(recall_score, pos_label=0),
    "f1": make_scorer(f1_score, pos_label=0),
}
cv = StratifiedKFold(n_splits=int(K), shuffle=True, random_state=42)
result = cross_validate(models[모델], X_train, y_train, cv=cv, scoring=scoring)

summary = pd.DataFrame({name: [result[f"test_{name}"].mean(), result[f"test_{name}"].std(),
                               ", ".join(f"{v:.3f}" for v in result[f"test_{name}"])] for name in scoring},
                       index=["평균", "표준편차", "폴드별"]).T
summary["평균"] = summary["평균"].astype(float).round(3)
summary["표준편차"] = summary["표준편차"].astype(float).round(3)
print(f"{모델}, {K}겹 층화 교차 검증 (학습 데이터 {len(y_train)}건)")
summary

**보고 방법**: "재현율 0.956 ± 0.025 (5겹 층화 교차 검증)". 표준편차가 크면 모델이 데이터에 따라 불안정하다는 뜻입니다.

품질 게이트는 평균에 걸되, **평균 − 표준편차**도 기준을 넘는지 함께 보면 더 보수적입니다.

## 4. [선택] 교차 검증에서도 누수는 생긴다

1일차 5교시의 무작위 데이터 실험을 다시 봅니다. 특성 선택을 교차 검증 **밖에서** 하면 누수가 생깁니다.

In [ ]:
#@title 4. [선택] 누수 실험
from sklearn.feature_selection import SelectKBest, f_classif

rng = np.random.RandomState(0)
X_noise = rng.normal(size=(100, 10000))
y_noise = rng.randint(0, 2, size=100)

leaky_X = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
leaky = cross_val_score(LogisticRegression(max_iter=1000), leaky_X, y_noise, cv=5).mean()
safe = cross_val_score(make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000)),
                       X_noise, y_noise, cv=5).mean()
print(f"누수: {leaky:.2f}  /  Pipeline: {safe:.2f}  (정답은 0.5 근처)")

## 5. 팀 과제

3번 칸에서 K와 모델을 바꿔 가며 아래 표를 채우고, **어떤 모델을 배포 후보로 올릴지** 한 문장으로 정하세요.

| 모델 | K | 재현율 평균 ± 표준편차 | 품질 게이트(0.90) 통과? |
| --- | --- | --- | --- |
| | | | |